# Gebäudesektor-Simulationen: XLSX → Parquet

**Prototyp – Schritt 1: Datenablage.** Dieses Notebook liest einen abgeschlossenen `building_sector`-Simulationslauf und schreibt die jährlichen PEExcel-Ein- und Ausgaben als Parquet-Dateien. Sämtlicher Konvertierungscode bleibt in diesem Notebook; es werden weder Excel COM noch Simulationen oder ein großer `Project` benötigt.

**Eingabe:** ein Ordner mit `batch.json` und `annual-*.xlsx` (jeweils mit den Blättern `IN` und `OUT`). Die Szenariospalten werden anhand des bestehenden Batch-Manifests identifiziert. Die originalen Dateien bleiben unverändert.

**Ausgabe:**

- `parquet/in.parquet` – **eine Zeile je Szenario**, IN-Variablen als Spalten
- `parquet/out.parquet` – **eine Zeile je Szenario**, OUT-Variablen als Spalten
- `parquet/conversion.json` – kompakte Herkunfts-/Typinformationen

Beide Tabellen enthalten die Identifikationsspalten `batch`, `scenario_name` und `source_file`. Gleiche Szenarionamen in *verschiedenen* Batches sind damit unterscheidbar. Innerhalb eines Batches müssen Szenarionamen eindeutig sein.

**Warum nicht `Project.from_files()`?** Das würde sämtliche `Scenario`-Objekte gleichzeitig aufbauen. Hier wird jeweils nur ein einzelner `annual`-Export gelesen und es werden kleine Gruppen direkt in Parquet geschrieben. Das umgehen wir auch das Excel-Spaltenlimit der späteren Gesamtausgabe.

> **Voraussetzungen:** `openpyxl` und `pyarrow` (ggf. einmalig im aktiven Notebook-Kernel mit `%pip install openpyxl pyarrow` installieren). Die XLSX-Dateien werden mit `data_only=True` gelesen: Formelergebnisse müssen daher bereits als Werte im Export gespeichert sein. Es findet keine Neuberechnung statt.

## 1. Konfiguration

Nur `SOURCE_DIR` muss für den nächsten Simulationslauf angepasst werden. Ein relativer Pfad bezieht sich auf das aktuelle Arbeitsverzeichnis des Notebook-Kernels (`Path.cwd()`); alternativ einen absoluten Windows-Pfad verwenden.

Die Verarbeitung umfasst bewusst nur `annual-*.xlsx`, damit vorhandene konsolidierte Excel-Dateien nicht erneut als Quellen eingelesen werden.

In [7]:
from __future__ import annotations

from collections import Counter, defaultdict
from datetime import date, datetime, time as datetime_time
import hashlib
import json
import logging
import math
from pathlib import Path
from time import perf_counter

import openpyxl
import pyarrow as pa
import pyarrow.parquet as pq

# Anpassen: relativer Pfad ab Notebook-Arbeitsverzeichnis oder absoluter Pfad
TYPOLOGY_NAME = "building_sector_v2"
SOURCE_DIR = Path("../../data/exports/typologies/", TYPOLOGY_NAME).resolve()
OUTPUT_DIR = Path("../../data/building_typologies", TYPOLOGY_NAME).resolve()

VERIFY_HASHES = False         # Optional: SHA256-Abgleich mit batch.json (zusätzliche I/O)
FILES_PER_ROW_GROUP = 8       # Mehr Quelldateien je Parquet-Row-Group = weniger Row-Groups
COMPRESSION = "zstd"
SHEETS = ("IN", "OUT")
ID_COLUMNS = ("batch", "scenario_name", "source_file")

logger = logging.getLogger("building_sector.parquet")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

print("Notebook working directory:", Path.cwd())
print("Source:", SOURCE_DIR)
print("Output:", OUTPUT_DIR)

Notebook working directory: c:\Users\schneids\code\peexcel\notebooks\Gebäudesektormodell
Source: C:\Users\schneids\code\peexcel\data\exports\typologies\building_sector_v2
Output: C:\Users\schneids\code\peexcel\data\building_typologies\building_sector_v2


## 2. Quellen anhand des Batch-Manifests auflösen

Das Manifest liefert für jede `annual-*.xlsx` die erwarteten Szenarionamen und verhindert, dass Metadatenspalten versehentlich als Szenarien interpretiert werden. Die Anzahl der Szenarien wird **nicht** fest vorgegeben. Abweichungen zwischen Dateibestand und Manifest führen zu einem Fehler.

In [8]:
def sha256(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def discover_sources(directory: Path):
    manifest_path = directory / "batch.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Batch-Manifest fehlt: {manifest_path}")

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("failed"):
        raise ValueError("Das Batch-Manifest enthält fehlgeschlagene Szenarien.")

    chunks = manifest.get("chunks", [])
    if not chunks:
        raise ValueError("Das Batch-Manifest enthält keine Chunks.")

    by_file = {chunk["annual"]: chunk for chunk in chunks}
    if len(by_file) != len(chunks):
        raise ValueError("Doppelte annual-Dateinamen im Manifest.")

    paths = sorted(p for p in directory.glob("annual-*.xlsx")
                   if not p.name.startswith("~$"))
    found = {p.name for p in paths}
    expected = set(by_file)
    if found != expected:
        raise ValueError(
            "Dateibestand unterscheidet sich vom Manifest: "
            f"fehlend={sorted(expected - found)}, "
            f"unerwartet={sorted(found - expected)}"
        )

    all_names = [name for chunk in chunks for name in chunk["scenarios"]]
    if not all_names or len(all_names) != len(set(all_names)):
        raise ValueError("Szenarionamen im Batch sind leer oder nicht eindeutig.")

    return manifest, paths, by_file, len(all_names)


manifest, source_files, chunks_by_file, expected_scenarios = discover_sources(SOURCE_DIR)
logger.info("Gefunden: %d XLSX-Dateien, %d Szenarien", len(source_files), expected_scenarios)

INFO Gefunden: 1296 XLSX-Dateien, 32400 Szenarien


## 3. Eine Excel-Datei in zwei tabellarische Blöcke lesen

PEExcel speichert die Variablen ursprünglich als **Zeilen** (`var_name`) und die Szenarien als **Spalten**. Für Parquet ist die Umkehrung zweckmäßiger: **Szenarien als Zeilen, Variablen als Spalten**.

Wir lesen ausschließlich die im Manifest genannten Szenariospalten. Die üblichen Metadaten (`Name`, `Einheit`, `Kommentar` usw.) werden nicht als Simulationsvariablen übernommen. Fehlende Werte bleiben `null`; Excel-Fehlerzellen lösen einen Fehler aus. Die Variablen werden **nicht** über das aktuelle `pexl`-Schema umbenannt, damit der Quellbestand unverändert repräsentiert bleibt.

In [9]:
def read_annual(path: Path, scenario_names: list[str]) -> dict[str, dict]:
    """Liest IN/OUT als {sheet: {var_name: [Werte pro Szenario]}}.

    Der Rückgabewert umfasst nur ein einzelnes Quelldokument im Speicher.
    """
    workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
    try:
        result = {}
        for sheet_name in SHEETS:
            if sheet_name not in workbook.sheetnames:
                raise ValueError(f"{path.name}: Tabellenblatt {sheet_name} fehlt")

            rows = iter(workbook[sheet_name].iter_rows())
            try:
                header = [cell.value for cell in next(rows)]
            except StopIteration as exc:
                raise ValueError(f"{path.name}/{sheet_name}: leeres Blatt") from exc

            while header and header[-1] is None:
                header.pop()
            if len(header) != len(set(header)) or "var_name" not in header:
                raise ValueError(f"{path.name}/{sheet_name}: ungültige oder doppelte Spaltennamen")

            missing = [name for name in scenario_names if name not in header]
            if missing:
                raise ValueError(f"{path.name}/{sheet_name}: Szenariospalten fehlen: {missing[:5]}")
            scenario_indices = [header.index(name) for name in scenario_names]
            if scenario_indices != sorted(scenario_indices):
                raise ValueError(f"{path.name}/{sheet_name}: Szenarioreihenfolge weicht vom Manifest ab")

            key_index = header.index("var_name")
            variables = {}
            for row in rows:
                key = row[key_index].value
                if key is None and all(cell.value is None for cell in row):
                    continue  # leere Abschlusszeilen
                if not isinstance(key, str) or not key:
                    raise ValueError(f"{path.name}/{sheet_name}: leere/ungültige var_name-Zeile")
                if key in variables or key in ID_COLUMNS:
                    raise ValueError(f"{path.name}/{sheet_name}: doppelte/reservierte Variable {key!r}")

                values = []
                for column_index in scenario_indices:
                    cell = row[column_index]
                    if cell.data_type == "e":
                        raise ValueError(f"{path.name}/{sheet_name}/{key}: Excel-Fehler {cell.value}")
                    value = cell.value
                    if isinstance(value, (int, float)) and not isinstance(value, bool):
                        if not math.isfinite(value):
                            raise ValueError(f"{path.name}/{sheet_name}/{key}: nichtendlicher Wert")
                    values.append(value)
                variables[key] = values

            result[sheet_name] = variables
        return result
    finally:
        workbook.close()

## 4. Schema bestimmen (erste, speicherschonende Lesepassage)

Parquet verwendet feste Datentypen je Spalte. Wir ermitteln deshalb vor dem Schreiben, welche Typen pro `var_name` über **alle** Quelldateien auftreten. So führen beispielsweise zunächst leere Werte nicht dazu, dass eine später numerische Variable fälschlich als reiner Null-Typ angelegt wird.

- Rein ganzzahlige Variablen → `int64`; numerische Variablen mit Dezimalzahlen → `float64`
- Boolean-Variablen → `bool`
- Text oder gemischter Text/Zahl-Inhalt → `string` (gemischte Typen werden protokolliert)

Das benötigt zwei Durchläufe über die XLSX-Dateien, aber zu keinem Zeitpunkt einen vollständigen `Project`. Für einen ersten, verlässlichen Prototyp ist diese Abwägung sinnvoll; später kann die Typdefinition aus dem PEExcel-Schema bezogen werden.

In [10]:
def value_kind(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, int):
        return "int"
    if isinstance(value, float):
        return "float"
    if isinstance(value, (date, datetime, datetime_time)):
        return "text"  # später ISO-Text
    return "text"


def arrow_type(kinds: set[str]) -> pa.DataType:
    if not kinds or "text" in kinds:
        return pa.string()
    if kinds == {"bool"}:
        return pa.bool_()
    if kinds == {"int"}:
        return pa.int64()
    return pa.float64()  # int + float, ggf. bool + numerisch


def inspect_types(paths, chunks):
    variable_order = {}
    kind_sets = {sheet: defaultdict(set) for sheet in SHEETS}
    sentinels = Counter()
    start = perf_counter()

    for index, path in enumerate(paths, start=1):
        chunk = chunks[path.name]
        if VERIFY_HASHES:
            actual = sha256(path)
            expected = chunk["files"][path.name]
            if actual != expected:
                raise ValueError(f"SHA256-Abweichung: {path.name}")

        tables = read_annual(path, chunk["scenarios"])
        for sheet, variables in tables.items():
            names = list(variables)
            if sheet in variable_order and names != variable_order[sheet]:
                raise ValueError(f"{path.name}/{sheet}: abweichende Variablenliste/Reihenfolge")
            variable_order.setdefault(sheet, names)
            for name, values in variables.items():
                for value in values:
                    kind = value_kind(value)
                    if kind is not None:
                        kind_sets[sheet][name].add(kind)
                    if (isinstance(value, (int, float)) and not isinstance(value, bool)
                            and -2146826300 <= value <= -2146826200):
                        sentinels[(sheet, name)] += 1

        if index % 10 == 0 or index == len(paths):
            logger.info("Schema geprüft: %d/%d Dateien", index, len(paths))

    logger.info("Schema-Passage abgeschlossen (%.1f s)", perf_counter() - start)
    return variable_order, kind_sets, sentinels


variable_order, kind_sets, sentinel_counts = inspect_types(source_files, chunks_by_file)

schemas = {}
for sheet in SHEETS:
    fields = [pa.field(name, pa.string()) for name in ID_COLUMNS]
    for name in variable_order[sheet]:
        kinds = kind_sets[sheet][name]
        if len(kinds) > 1 and ("text" in kinds or "bool" in kinds):
            logger.warning("Gemischte Datentypen bei %s/%s: %s", sheet, name, sorted(kinds))
        fields.append(pa.field(name, arrow_type(kinds)))
    schemas[sheet] = pa.schema(fields)
    logger.info("%s: %d Variablen", sheet, len(variable_order[sheet]))

if sentinel_counts:
    logger.warning("Excel-Fehlersentinels gefunden (als Quellwerte beibehalten): %d", sum(sentinel_counts.values()))

INFO Schema geprüft: 10/1296 Dateien
INFO Schema geprüft: 20/1296 Dateien
INFO Schema geprüft: 30/1296 Dateien
INFO Schema geprüft: 40/1296 Dateien
INFO Schema geprüft: 50/1296 Dateien
INFO Schema geprüft: 60/1296 Dateien
INFO Schema geprüft: 70/1296 Dateien
INFO Schema geprüft: 80/1296 Dateien
INFO Schema geprüft: 90/1296 Dateien
INFO Schema geprüft: 100/1296 Dateien
INFO Schema geprüft: 110/1296 Dateien
INFO Schema geprüft: 120/1296 Dateien
INFO Schema geprüft: 130/1296 Dateien
INFO Schema geprüft: 140/1296 Dateien
INFO Schema geprüft: 150/1296 Dateien
INFO Schema geprüft: 160/1296 Dateien
INFO Schema geprüft: 170/1296 Dateien
INFO Schema geprüft: 180/1296 Dateien
INFO Schema geprüft: 190/1296 Dateien
INFO Schema geprüft: 200/1296 Dateien
INFO Schema geprüft: 210/1296 Dateien
INFO Schema geprüft: 220/1296 Dateien
INFO Schema geprüft: 230/1296 Dateien
INFO Schema geprüft: 240/1296 Dateien
INFO Schema geprüft: 250/1296 Dateien
INFO Schema geprüft: 260/1296 Dateien
INFO Schema geprüft: 

## 5. Parquet schreiben (zweite Lesepassage)

Der `ParquetWriter` hängt pro Gruppe von Quelldateien eine **Row Group** an dieselbe Zieldatei an. Dadurch entsteht pro Tabellenblatt genau **eine** Parquet-Datei, ohne 32.400 oder mehr `Scenario`-Objekte zu erzeugen. `FILES_PER_ROW_GROUP` steuert die Zwischenspeichergröße.

Die Ausgaben werden zunächst unter `.pending.parquet` geschrieben und erst nach erfolgreichem Schreiben und Prüfung ihrer Zeilenzahlen umbenannt. Das vermeidet unvollständige Dateien mit den endgültigen Namen.

In [13]:
def convert_value(value, dtype: pa.DataType):
    if value is None:
        return None
    if pa.types.is_string(dtype):
        if isinstance(value, (date, datetime, datetime_time)):
            return value.isoformat()
        return value if isinstance(value, str) else str(value)
    if pa.types.is_boolean(dtype):
        return bool(value)
    if pa.types.is_integer(dtype):
        return int(value)
    if pa.types.is_floating(dtype):
        return float(value)
    raise TypeError(f"Nicht unterstützter Arrow-Typ: {dtype}")


# Gemeinsames Schema: Metadaten + alle IN- und OUT-Variablen
META_COLUMNS = ("batch", "scenario_name", "source_file")

combined_schema = pa.schema(
    [schemas["IN"].field(name) for name in META_COLUMNS]
    + [
        schemas[sheet].field(name)
        for sheet in SHEETS
        for name in variable_order[sheet]
    ]
)

# Eindeutigkeit der Spalten prüfen
column_names = combined_schema.names
if len(column_names) != len(set(column_names)):
    raise ValueError("Doppelte Spaltennamen im kombinierten Schema")


def arrow_table(tables, names, filename, batch_id):
    """Combine IN and OUT into one row per scenario."""
    data = {
        "batch": [batch_id] * len(names),
        "scenario_name": names,
        "source_file": [filename] * len(names),
    }

    for sheet in SHEETS:
        variables = tables[sheet]

        if list(variables) != variable_order[sheet]:
            raise ValueError(
                f"Variablenliste geändert: {filename}/{sheet}"
            )

        for name in variable_order[sheet]:
            dtype = combined_schema.field(name).type
            data[name] = [
                convert_value(value, dtype)
                for value in variables[name]
            ]

    return pa.Table.from_pydict(data, schema=combined_schema)


def write_parquet(paths, chunks, output_dir: Path, batch_id: str):
    """Write all scenarios into a single Parquet file."""
    output_dir.mkdir(parents=True, exist_ok=True)

    target = output_dir / "scenarios.parquet"
    pending = output_dir / "scenarios.pending.parquet"

    buffer = []
    written = 0
    started = perf_counter()

    try:
        with pq.ParquetWriter(
            pending,
            combined_schema,
            compression=COMPRESSION,
            use_dictionary=True,
        ) as writer:

            for index, path in enumerate(paths, start=1):
                names = chunks[path.name]["scenarios"]
                tables = read_annual(path, names)

                buffer.append(
                    arrow_table(tables, names, path.name, batch_id)
                )

                if index % FILES_PER_ROW_GROUP == 0 or index == len(paths):
                    combined = pa.concat_tables(buffer)
                    writer.write_table(combined)

                    written += combined.num_rows
                    buffer.clear()

                    logger.info(
                        "Parquet geschrieben: %d/%d Dateien (%d Szenarien)",
                        index, len(paths), written,
                    )

        # Verify before publishing the finished file
        actual = pq.ParquetFile(pending).metadata.num_rows

        if actual != expected_scenarios:
            raise ValueError(
                f"{actual} statt {expected_scenarios} Szenarien"
            )

        pending.replace(target)

    except Exception:
        try:
            pending.unlink(missing_ok=True)
        except OSError:
            logger.warning("Temporäre Datei konnte nicht entfernt werden: %s", pending)
        raise

    logger.info(
        "Fertig in %.1f s: %s",
        perf_counter() - started,
        target,
    )

    return target, written


if FILES_PER_ROW_GROUP < 1:
    raise ValueError("FILES_PER_ROW_GROUP muss mindestens 1 sein")

output_file, written = write_parquet(
    source_files,
    chunks_by_file,
    OUTPUT_DIR,
    batch_id=SOURCE_DIR.name,
)

INFO Parquet geschrieben: 8/1296 Dateien (200 Szenarien)
INFO Parquet geschrieben: 16/1296 Dateien (400 Szenarien)
INFO Parquet geschrieben: 24/1296 Dateien (600 Szenarien)
INFO Parquet geschrieben: 32/1296 Dateien (800 Szenarien)
INFO Parquet geschrieben: 40/1296 Dateien (1000 Szenarien)
INFO Parquet geschrieben: 48/1296 Dateien (1200 Szenarien)
INFO Parquet geschrieben: 56/1296 Dateien (1400 Szenarien)
INFO Parquet geschrieben: 64/1296 Dateien (1600 Szenarien)
INFO Parquet geschrieben: 72/1296 Dateien (1800 Szenarien)
INFO Parquet geschrieben: 80/1296 Dateien (2000 Szenarien)
INFO Parquet geschrieben: 88/1296 Dateien (2200 Szenarien)
INFO Parquet geschrieben: 96/1296 Dateien (2400 Szenarien)
INFO Parquet geschrieben: 104/1296 Dateien (2600 Szenarien)
INFO Parquet geschrieben: 112/1296 Dateien (2800 Szenarien)
INFO Parquet geschrieben: 120/1296 Dateien (3000 Szenarien)
INFO Parquet geschrieben: 128/1296 Dateien (3200 Szenarien)
INFO Parquet geschrieben: 136/1296 Dateien (3400 Szenarie

## 6. Ergebnis prüfen und Herkunft dokumentieren

Wir prüfen die tatsächlichen Parquet-Zeilenzahlen und zeigen einige Spalten. Für spätere Batch-Vergleiche bleiben der Batchname, die Quell-Datei und das Original-Szenario als separate Spalten erhalten.

`conversion.json` dokumentiert die Anzahl Szenarien, Quelldateien, Variablen und den verwendeten Datentyp pro Variable. Die Datei ist **keine** vollständige Simulationsevaluierung.

In [14]:
report = {
    "batch": SOURCE_DIR.name,
    "source_directory": str(SOURCE_DIR),
    "scenario_count": expected_scenarios,
    "source_file_count": len(source_files),
    "hashes_verified": VERIFY_HASHES,
    "output": {
        "file": output_file.name,
        "rows": written,
        "columns": len(combined_schema.names),
        "variables": {
            sheet: len(variable_order[sheet])
            for sheet in SHEETS
        },
    },
    "types": {
        sheet: {
            name: str(schemas[sheet].field(name).type)
            for name in variable_order[sheet]
        }
        for sheet in SHEETS
    },
    "excel_error_sentinels": {
        f"{sheet}/{name}": count
        for (sheet, name), count in sentinel_counts.items()
    },
}

# Write conversion report
report_path = OUTPUT_DIR / "conversion.json"
report_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# Inspect resulting Parquet file
pf = pq.ParquetFile(output_file)

print(
    f"\n{pf.metadata.num_rows:,} scenarios · "
    f"{len(pf.schema_arrow.names)} columns · "
    f"{output_file.stat().st_size / 1024**2:.1f} MiB"
)

print("Variables:", {
    sheet: len(variable_order[sheet])
    for sheet in SHEETS
})

# Sample: metadata + first 3 variables from IN and OUT
cols = list(META_COLUMNS) + [
    name
    for sheet in SHEETS
    for name in variable_order[sheet][:3]
]

sample = next(
    pf.iter_batches(batch_size=3, columns=cols)
).to_pydict()

print("\nSample columns:", cols)

for i in range(len(sample["scenario_name"])):
    print(" ", {
        name: sample[name][i]
        for name in cols
    })

print("\nReport:", report_path)


32,400 scenarios · 1213 columns · 59.4 MiB
Variables: {'IN': 698, 'OUT': 512}

Sample columns: ['batch', 'scenario_name', 'source_file', 'tool_version', 'input_version_number', 'cfd_fPE', 'Ta_annual_avg', 'Ta_avg_heating_period', 'Ta_avg_cooling_period']
  {'batch': 'building_sector_v2', 'scenario_name': 'EFH_1G|vor1919|WP_Sole|PV30|noFlex|Wien |Leicht', 'source_file': 'annual-00001.xlsx', 'tool_version': '1.13.1_dev', 'input_version_number': '9', 'cfd_fPE': 1.63, 'Ta_annual_avg': 11.05647260273968, 'Ta_avg_heating_period': 7.051928374655672, 'Ta_avg_cooling_period': -2146826281.0}
  {'batch': 'building_sector_v2', 'scenario_name': 'EFH_1G|vor1919|WP_Sole|PV30|noFlex|Wien |mittel', 'source_file': 'annual-00001.xlsx', 'tool_version': '1.13.1_dev', 'input_version_number': '9', 'cfd_fPE': 1.63, 'Ta_annual_avg': 11.05647260273968, 'Ta_avg_heating_period': 7.051928374655672, 'Ta_avg_cooling_period': -2146826281.0}
  {'batch': 'building_sector_v2', 'scenario_name': 'EFH_1G|vor1919|WP_Sole|P

In [23]:
import pyarrow.parquet as pq
import pandas as pd
from pexl import glossary

In [27]:

parquet = pq.ParquetFile(output_file)

print(f"Szenarien: {parquet.metadata.num_rows:,}")
print(f"Variablen + Metadaten: {parquet.metadata.num_columns:,}")
print(f"Row Groups: {parquet.metadata.num_row_groups:,}")

df = pd.read_parquet(
    output_file,
    columns=[
        "scenario_name",
        glossary.preset_recorded_geometry.var_name,
        glossary.PV_roof_share.var_name,
        glossary.PEI_balance.var_name,
    ],
)

display(df.head())

Szenarien: 32,400
Variablen + Metadaten: 1,213
Row Groups: 162


,scenario_name,preset_recorded_geometry,PV_roof_share,PEI_balance
0,EFH_1G|vor1919|WP_Sole|PV30|noFlex|Wien |Leicht,EFH_1G,0.3,-194.080928
1,EFH_1G|vor1919|WP_Sole|PV30|noFlex|Wien |mittel,EFH_1G,0.3,-194.067660
2,EFH_1G|vor1919|WP_Sole|PV30|noFlex|Wien |schwer,EFH_1G,0.3,-194.130217
3,EFH_1G|vor1919|WP_Sole|PV30|noFlex|Innsbruck |...,EFH_1G,0.3,-208.534764
4,EFH_1G|vor1919|WP_Sole|PV30|noFlex|Innsbruck |...,EFH_1G,0.3,-208.438723


## Ergebnis und nächster Schritt

Die beiden Dateien sind eine **verlustarme, schema-agnostische Ablage der gespeicherten IN-/OUT-Zellwerte**; die Metadatenzeilen der ursprünglichen Excel-Dateien sind nicht Teil der Parquet-Tabellen. Gemischte Datentypen können Textspalten ergeben und werden beim Konvertieren protokolliert. Der Prototyp setzt konsistente Variablenlisten innerhalb eines Batches voraus; unterschiedliche Schemaversionen *zwischen* Batches sind ein eigener Schritt.

Später können wir die verschiedenen Batch-Parquet-Dateien mit DuckDB oder PyArrow **gemeinsam abfragen**, ohne daraus zunächst ein riesiges `Project` zu erzeugen. Für die Detailanalyse lässt sich anschließend eine kleine Szenarioauswahl wieder als `Project` materialisieren. Diese weiteren Schritte sind hier noch nicht implementiert.